# Q2: Hybrid Intelligent System — Fuzzy Logic + Neural Network
**Inputs:** Attendance (%), Assignment Marks, Test Marks  
**Output:** Performance Level (Poor, Average, Good)  



In [ ]:
# Step 1: Install required libraries
!pip install scikit-fuzzy tensorflow

In [ ]:
# Step 2: Import Libraries
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)
print('All libraries imported successfully!')

In [ ]:
# Step 3: Generate Synthetic Student Dataset (100 students)
n = 100
attendance      = np.random.randint(40, 101, n)   # 40–100%
assignment_marks = np.random.randint(30, 101, n)  # 30–100
test_marks      = np.random.randint(30, 101, n)   # 30–100

# Rule-based labels for ground truth
def label_student(a, am, tm):
    avg = (a * 0.3 + am * 0.3 + tm * 0.4)
    if avg >= 70:   return 'Good'
    elif avg >= 50: return 'Average'
    else:           return 'Poor'

labels = [label_student(a, am, tm)
          for a, am, tm in zip(attendance, assignment_marks, test_marks)]

df = pd.DataFrame({
    'Attendance':       attendance,
    'Assignment_Marks': assignment_marks,
    'Test_Marks':       test_marks,
    'Performance':      labels
})

print(df.head(10))
print('\nClass distribution:')
print(df['Performance'].value_counts())

In [ ]:
# Step 4: FUZZY LAYER — Define membership functions

att  = ctrl.Antecedent(np.arange(0, 101, 1), 'attendance')
am   = ctrl.Antecedent(np.arange(0, 101, 1), 'assignment_marks')
tm   = ctrl.Antecedent(np.arange(0, 101, 1), 'test_marks')
perf = ctrl.Consequent(np.arange(0, 101, 1), 'performance')

# Attendance MFs
att['low']    = fuzz.trimf(att.universe, [0,  0,  60])
att['medium'] = fuzz.trimf(att.universe, [50, 70, 85])
att['high']   = fuzz.trimf(att.universe, [75, 100, 100])

# Assignment Marks MFs
am['low']    = fuzz.trimf(am.universe, [0,  0,  50])
am['medium'] = fuzz.trimf(am.universe, [40, 60, 80])
am['high']   = fuzz.trimf(am.universe, [70, 100, 100])

# Test Marks MFs
tm['low']    = fuzz.trimf(tm.universe, [0,  0,  50])
tm['medium'] = fuzz.trimf(tm.universe, [40, 60, 80])
tm['high']   = fuzz.trimf(tm.universe, [70, 100, 100])

# Performance output MFs
perf['poor']    = fuzz.trimf(perf.universe, [0,  0,  40])
perf['average'] = fuzz.trimf(perf.universe, [30, 50, 70])
perf['good']    = fuzz.trimf(perf.universe, [60, 100, 100])

print('Fuzzy membership functions defined!')

In [ ]:
# Step 5: Plot all membership functions
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

att.view(ax=axes[0][0]);  axes[0][0].set_title('Attendance')
am.view(ax=axes[0][1]);   axes[0][1].set_title('Assignment Marks')
tm.view(ax=axes[1][0]);   axes[1][0].set_title('Test Marks')
perf.view(ax=axes[1][1]); axes[1][1].set_title('Performance Output')

plt.suptitle('Fuzzy Membership Functions — Student Performance System', fontsize=13)
plt.tight_layout()
plt.savefig('q2_membership_functions.png', dpi=150)
plt.show()

In [ ]:
# Step 6: Define Fuzzy Rules
rules = [
    ctrl.Rule(att['high']   & am['high']   & tm['high'],   perf['good']),
    ctrl.Rule(att['high']   & am['medium'] & tm['high'],   perf['good']),
    ctrl.Rule(att['medium'] & am['medium'] & tm['medium'], perf['average']),
    ctrl.Rule(att['medium'] & am['high']   & tm['medium'], perf['average']),
    ctrl.Rule(att['low']    & am['low']    & tm['low'],    perf['poor']),
    ctrl.Rule(att['low']    & am['medium'] & tm['low'],    perf['poor']),
    ctrl.Rule(att['high']   & am['low']    & tm['medium'], perf['average']),
    ctrl.Rule(att['medium'] & am['low']    & tm['low'],    perf['poor']),
]

student_ctrl = ctrl.ControlSystem(rules)
student_sim  = ctrl.ControlSystemSimulation(student_ctrl)
print(f'{len(rules)} fuzzy rules created!')

In [ ]:
# Step 7: Apply fuzzy system to generate fuzzy scores for each student
fuzzy_scores = []

for _, row in df.iterrows():
    try:
        student_sim.input['attendance']       = row['Attendance']
        student_sim.input['assignment_marks'] = row['Assignment_Marks']
        student_sim.input['test_marks']       = row['Test_Marks']
        student_sim.compute()
        fuzzy_scores.append(student_sim.output['performance'])
    except:
        fuzzy_scores.append(50.0)   # fallback default

df['Fuzzy_Score'] = fuzzy_scores
print('Fuzzy scores computed for all students!')
print(df[['Attendance','Assignment_Marks','Test_Marks','Fuzzy_Score','Performance']].head(10))

In [ ]:
# Step 8: NEURAL NETWORK LAYER
# Features: original inputs + fuzzy score (hybrid combination)
X = df[['Attendance', 'Assignment_Marks', 'Test_Marks', 'Fuzzy_Score']].values / 100.0

le = LabelEncoder()
y_encoded = le.fit_transform(df['Performance'])  # Poor=0, Average=1, Good=2... sorted
y_onehot  = keras.utils.to_categorical(y_encoded, num_classes=3)

X_train, X_test, y_train, y_test, ye_train, ye_test = train_test_split(
    X, y_onehot, y_encoded, test_size=0.2, random_state=42
)

print('Classes:', le.classes_)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

In [ ]:
# Step 9: Build Neural Network
model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(4,), name='Hidden_1'),
    keras.layers.Dense(8,  activation='relu', name='Hidden_2'),
    keras.layers.Dense(3,  activation='softmax', name='Output')  # 3 classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Step 10: Train the Neural Network
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.1,
    verbose=1
)
print('Training complete!')

In [ ]:
# Step 11: Test with a new student
def predict_student(attendance_val, assignment_val, test_val):
    # Fuzzy layer
    try:
        student_sim.input['attendance']       = attendance_val
        student_sim.input['assignment_marks'] = assignment_val
        student_sim.input['test_marks']       = test_val
        student_sim.compute()
        fuzzy_score = student_sim.output['performance']
    except:
        fuzzy_score = 50.0

    # Neural network layer
    features = np.array([[attendance_val, assignment_val, test_val, fuzzy_score]]) / 100.0
    probs    = model.predict(features, verbose=0)[0]
    pred_idx = np.argmax(probs)

    print(f'\nStudent Input:')
    print(f'  Attendance:       {attendance_val}%')
    print(f'  Assignment Marks: {assignment_val}')
    print(f'  Test Marks:       {test_val}')
    print(f'  Fuzzy Score:      {fuzzy_score:.2f}')
    print(f'\nPrediction Probabilities:')
    for cls, prob in zip(le.classes_, probs):
        print(f'  {cls}: {prob*100:.1f}%')
    print(f'\n==> Predicted Performance: {le.classes_[pred_idx].upper()}')

# Test it!
predict_student(attendance_val=85, assignment_val=78, test_val=82)
predict_student(attendance_val=55, assignment_val=45, test_val=40)
predict_student(attendance_val=75, assignment_val=88, test_val=100)

## How Fuzzy Logic and Neural Network are Integrated

### Fuzzy Layer (Rule-Based Reasoning)
- Handles **linguistic uncertainty** in student data
- Converts crisp numeric inputs into **fuzzy membership degrees** (e.g., attendance=75% is 50% 'medium' and 50% 'high')
- Applies **IF-THEN rules** to generate a fuzzy performance score (0–100)
- This mimics how a teacher would intuitively evaluate a student

### Neural Network Layer (Learning)
- Takes both **original inputs + fuzzy score** as features
- Learns **non-linear patterns** in data that fuzzy rules might miss
- Uses **backpropagation** to tune weights automatically from training data
- Outputs probability for each class: Poor, Average, Good

### Why Hybrid?
| Fuzzy Logic | Neural Network |
|------------|----------------|
| Interpretable rules | Learns from data automatically |
| Handles vague/uncertain input | Handles complex non-linear patterns |
| No training data needed | Improves with more data |
| Expert knowledge encoded | Adapts to new patterns |

The hybrid system combines **the best of both worlds**.